<a href="https://colab.research.google.com/github/heberdavi/mba-engsoft-tcc/blob/evolucao-pos-tcc/notebooks/01-processamento_pln.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

📑 Guia de Execução
⚠️ IMPORTANTE: Sempre que o Runtime (Ambiente de Execução) for reiniciado, as Células 1 e 2 devem ser executadas obrigatoriamente para restabelecer os caminhos do Drive e reinstalar as bibliotecas.

🔄 Fluxo de Dependências:
Sessão Recém-Iniciada: Executar Célula 1 ➔ Célula 2.

Primeira vez no projeto: Executar Célula 1 ➔ Célula 2 ➔ Célula 3 (Carga).

Retomando Processamento: Se o banco já existe no Drive, pule a Célula 3 e vá direto para a Célula 3.1 e, em seguida, Célula 4 e/ou 5 e/ou 6.

⚠️ Para a correta execução da Célula 1, é necessário observar a estrutura de subpastas no Google Drive, dentro de uma pasta principal. Atualmente, a estrutura principal é **..\pln\evolucao-pos-tcc** e deve ser armazenada na raíz do Google Drive (MyDrive)):
*   deve haver uma subpasta chamada **\data**, onde será criado automaticamente o banco SQLite (um arquivo com a extensão .db, denominado **base-dados.db**);
*   deve haver uma subpasta chamada **\sql**, onde deverão ser adicionados os dois arquivos .sql responsáveis pela criação das tabelas e população dos dados no banco SQLite. Os arquivos podem ser obtidos junto ao projeto no Github, na pasta **\sql**, sendo  **01-schema.sql** e **02-seed_data.sql**.



In [ ]:
# Célula 1: Montagem do Google Drive e Configuração de Caminhos
from google.colab import drive
import os

# 1. Montagem Segura: Só executa se ainda não estiver montado
if not os.path.exists('/content/drive/MyDrive'):
    print("📂 Montando Google Drive...")
    drive.mount('/content/drive')
else:
    print("✅ Google Drive já está montado e acessível.")

# 2. Configuração Estrita de Caminhos
DRIVE_DIR = '/content/drive/MyDrive/pln/evolucao-pos-tcc'
DB_FILE_NAME = 'data/base-dados.db'
DB_PATH = os.path.join(DRIVE_DIR, DB_FILE_NAME)

# Artefatos SQL
SCHEMA_SQL = os.path.join(DRIVE_DIR, 'sql/01-schema.sql')
SEED_SQL = os.path.join(DRIVE_DIR, 'sql/02-seed_data.sql')

# Pasta de Saída (Outputs)
EXPORT_PATH = os.path.join(DRIVE_DIR, 'outputs')
if not os.path.exists(EXPORT_PATH):
    os.makedirs(EXPORT_PATH)
    print(f"📁 Pasta de exportação criada em: {EXPORT_PATH}")

print(f"📍 Banco de Dados: {DB_PATH}")

⚠️ Na Célula 2 ocorre a criação das estruturas (tabelas, colunas, etc) no banco de dados SQLite.

In [ ]:
# Célula 2: Instalação das bibliotecas e inicialização da estrutura (Schema)

# 1. Instalação Silenciosa
!pip install -q transformers torch pandas bertopic pysentimiento spacy
!python -m spacy download pt_core_news_lg -q

import sqlite3
import torch

# 2. Hardware Check para BERTimbau/BERTopic
device = 0 if torch.cuda.is_available() else -1

def inicializar_estrutura_db(db_path, schema_path):
    """Garante que a estrutura de tabelas esteja presente."""
    print(f"🛠️ Verificando integridade das tabelas...")

    # Se o arquivo de banco não existir, o SQLite o criará automaticamente
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        with open(schema_path, 'r', encoding='utf-8') as f:
            cursor.executescript(f.read())
        conn.commit()
        print("✅ Estrutura (Schema) validada com sucesso!")
    except Exception as e:
        print(f"❌ Erro ao processar Schema: {e}")
    finally:
        conn.close()

# 3. Execução
inicializar_estrutura_db(DB_PATH, SCHEMA_SQL)

print(f"\n🚀 Ambiente pronto (GPU: {'Ativa' if device == 0 else 'Inativa'}).")

⚠️ Na Célula 3, os dados são populados no banco de dados SQLite.

In [ ]:
# Célula 3: Carga Inicial de Dados (Seed SQL)
def executar_carga_dados(db_path, seed_path):
    """Popula o banco apenas se a tabela 'verso' estiver vazia."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        # Verifica se já existem dados para evitar duplicidade no Drive
        cursor.execute("SELECT count(*) FROM verso")
        total_existente = cursor.fetchone()[0]

        if total_existente > 0:
            print(f"ℹ️ O banco já contém {total_existente} versos. Carga inicial ignorada.")
            return

        print("🌱 Semeando dados iniciais (02-seed_data.sql)... Isso pode levar alguns minutos.")
        with open(seed_path, 'r', encoding='utf-8') as f:
            cursor.executescript(f.read())

        conn.commit()
        print(f"✅ Carga de {seed_path} concluída com sucesso!")

    except sqlite3.OperationalError as e:
        print(f"⚠️ Erro operacional: {e}. Verifique se a Célula 2 foi executada.")
    except Exception as e:
        print(f"❌ Erro crítico na carga: {e}")
    finally:
        conn.close()

# Executa a carga (Somente se necessário)
executar_carga_dados(DB_PATH, SEED_SQL)

In [ ]:
# Célula 3.1: Registro da Execução do Pipeline (Com Captura Dinâmica)
import sqlite3
from datetime import datetime

def registrar_execucao_pipeline(conn_db, observacao: str) -> int:
    """
    Insere um novo registro na tabela execucao_pipeline e retorna o ID gerado.
    """
    cursor = conn_db.cursor()
    query = """
        INSERT INTO execucao_pipeline (observacao)
        VALUES (?);
    """
    cursor.execute(query, (observacao,))
    conn_db.commit()

    exec_id = cursor.lastrowid
    print(f"\n✓ Nova execução registrada com sucesso!")
    print(f"  └─ Execução ID: {exec_id}")
    print(f"  └─ Observação: {observacao}\n")

    return exec_id

# 1. Captura dinâmica do texto de observação via prompt do usuário
obs_usuario = input("📝 Digite a observação para esta execução do pipeline (ou aperte Enter para o padrão): ").strip()

# 2. Define um texto padrão caso o usuário não digite nada
if not obs_usuario:
    OBSERVACAO_EXECUCAO = "Execução Zero-Shot com versão vigente de tópicos e tratamento de ruído."
else:
    OBSERVACAO_EXECUCAO = obs_usuario

# 3. Conexão e Registro
try:
    conn = sqlite3.connect(DB_PATH)

    # Registra e define a variável global que será consumida pelas células 4, 5 e 6
    EXECUCAO_PIPELINE_ID = registrar_execucao_pipeline(conn, OBSERVACAO_EXECUCAO)

except NameError:
    print("❌ Erro: Variável 'DB_PATH' não encontrada. Certifique-se de executar as células de configuração primeiro.")
except sqlite3.Error as e:
    print(f"❌ Erro ao registrar execução no banco de dados: {e}")
finally:
    if 'conn' in locals() and conn:
        conn.close()

⚠️ Nas Células 4, 5 e 6 ocorre o processamento do corpus bíblico, verso por verso, e a armazenagem dos dados nas tabelas que registram os detalhes do processamento. Nesta versão do pipeline, a cada execução, as tabelas são limpas e novos dados são inseridos. Não há, portanto, um histórico de execuções que permita comparar os resultados caso haja mudanças nas regras e/ou lógica do processamento.

In [ ]:
# Célula 4: Limpeza Estrutural e Filtro de Densidade (Antídoto ao Ruído Nominal)
import spacy
import sqlite3
import pandas as pd
import re

# Carrega o modelo de português
try:
    nlp = spacy.load("pt_core_news_lg")
except:
    !python -m spacy download pt_core_news_lg
    nlp = spacy.load("pt_core_news_lg")

def limpar_texto_estrutural(texto):
    if not texto or len(texto.strip()) < 3: return "RUIDO_CURTO"

    doc = nlp(texto)

    # Filtramos tokens válidos (substantivos, verbos, adjetivos e nomes próprios)
    # Ignoramos stop words e pontuação
    tokens = [t for t in doc if not t.is_stop and not t.is_punct and t.pos_ in ['NOUN', 'VERB', 'ADJ', 'PROPN']]

    if not tokens: return "RUIDO_VAZIO"

    # Métrica 1: Densidade de Nomes Próprios (PROPN)
    # Se mais de 70% do conteúdo significativo forem nomes próprios, é provavelmente uma lista/genealogia
    propn_count = len([t for t in tokens if t.pos_ == 'PROPN'])
    propn_ratio = propn_count / len(tokens)

    # Métrica 2: Presença de Ação/Estado
    # Antídotos existenciais raramente são frases sem verbos ou adjetivos
    has_action_or_state = any(t.pos_ in ['VERB', 'ADJ'] for t in tokens)

    # CASO CRÍTICO (Ex: Nesias e Hatifa):
    # Verso curto, alta proporção de nomes próprios e sem verbo
    if len(tokens) <= 3 and propn_ratio > 0.5 and not has_action_or_state:
        return "RUIDO_NOMINAL"

    # Retornamos o texto limpo (usando o texto original para preservar a semântica)
    return " ".join([t.text.lower() for t in tokens])

# Execução e Persistência
conn = sqlite3.connect(DB_PATH)

# Garante que a execução ativa está acessível
try:
    exec_id = EXECUCAO_PIPELINE_ID
except NameError:
    raise RuntimeError("❌ EXECUCAO_PIPELINE_ID não encontrada. Execute a Célula 3.1 antes de rodar a Célula 4.")

print(f"🧼 Limpando textos para a Execução Pipeline ID: {exec_id}...")
df_versos = pd.read_sql_query("SELECT id AS verso_id, texto FROM verso", conn)

# Aplica a função de limpeza aos textos dos versos
df_versos['texto_limpo'] = df_versos['texto'].apply(limpar_texto_estrutural)

# Prepara a lista de tuplas para inserção em lote (verso_id, execucao_pipeline_id, texto_limpo)
dados_insercao = [
    (int(row['verso_id']), int(exec_id), str(row['texto_limpo']))
    for _, row in df_versos.iterrows()
]

# UPSERT SQL: Insere novos registros ou atualiza caso reprocessado dentro da MESMA execução
query_upsert = """
    INSERT INTO verso_limpo (verso_id, execucao_pipeline_id, texto_limpo)
    VALUES (?, ?, ?)
    ON CONFLICT(verso_id, execucao_pipeline_id) DO UPDATE SET
        texto_limpo = excluded.texto_limpo;
"""

cursor = conn.cursor()
cursor.executemany(query_upsert, dados_insercao)

conn.commit()
conn.close()

print(f"✅ Célula 4 concluída! {len(dados_insercao)} versos processados e gravados em 'verso_limpo' para a execução #{exec_id}.")

In [ ]:
# Célula 5: Classificação por Eixos Existenciais (Zero-Shot) com Persistência Dinâmica N-Tópicos
from transformers import pipeline
import sqlite3
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

# 1. Validação do Contexto da Execução Atual
try:
    exec_id = EXECUCAO_PIPELINE_ID
except NameError:
    raise RuntimeError("❌ EXECUCAO_PIPELINE_ID não encontrada. Execute a Célula 3.1 antes de rodar a Célula 5.")

conn = sqlite3.connect(DB_PATH)

# 2. Carga Dinâmica dos Tópicos, Descrições Vigentes e Identificação do Descarte
print("📖 Carregando descrições/hipóteses vigentes da VIEW 'v_topico_descricao_vigente'...")
query_topicos = """
    SELECT
        v.topico_descricao_id,
        v.topico_id,
        v.topico_nome,
        v.descricao,
        t.eh_descarte
    FROM v_topico_descricao_vigente v
    JOIN topico t ON v.topico_id = t.id
    ORDER BY t.id ASC;
"""
df_topicos = pd.read_sql_query(query_topicos, conn)

if df_topicos.empty:
    conn.close()
    raise ValueError("❌ Nenhuma descrição cadastrada na tabela 'topico_descricao'. Verifique a carga de seed data.")

# Captura listas ordenadas de descrições e IDs
descricoes_eixos = df_topicos['descricao'].tolist()
topicos_desc_ids = df_topicos['topico_descricao_id'].tolist()
num_topicos = len(descricoes_eixos)

# Localiza o ID da descrição associada ao tópico marcado com eh_descarte = 1
df_descarte = df_topicos[df_topicos['eh_descarte'] == 1]
if df_descarte.empty:
    conn.close()
    raise ValueError("❌ Nenhum tópico marcado com 'eh_descarte = 1' na tabela 'topico'. Ajuste a flag no banco de dados.")

topico_desc_id_descarte = int(df_descarte.iloc[0]['topico_descricao_id'])
idx_descarte_matriz = df_topicos.index[df_topicos['eh_descarte'] == 1].tolist()[0]

print(f"✓ {num_topicos} eixos conceituais carregados dinamicamente do banco.")
print(f"✓ Tópico de descarte identificado: '{df_descarte.iloc[0]['topico_nome']}' (topico_descricao_id={topico_desc_id_descarte}).")

# 3. Carga dos Versos e do Texto Limpo da Execução Atual
query_versos = """
    SELECT v.id AS verso_id, v.texto, l.abreviacao, g.id AS genero_id, vl.texto_limpo
    FROM verso v
    JOIN verso_limpo vl ON v.id = vl.verso_id
    JOIN livro l ON l.id = v.livro_id
    JOIN genero_literario g ON g.id = l.genero_id
    WHERE vl.execucao_pipeline_id = ?;
"""
df_input = pd.read_sql_query(query_versos, conn, params=(exec_id,))

# Tratamento de segurança contra valores nulos
df_input['texto'] = df_input['texto'].fillna('vazio').astype(str)
df_input['texto_limpo'] = df_input['texto_limpo'].fillna('RUIDO_VAZIO').astype(str)

docs_para_classificar = [str(doc).strip() if (doc and str(doc).strip() != "") else "vazio" for doc in df_input['texto'].tolist()]

# 4. Inicialização do Classificador Zero-Shot Nativo (Hugging Face / BERTimbau)
print("🚀 Inicializando classificador Zero-Shot (neuralmind/bert-base-portuguese-cased)...")
classifier = pipeline(
    "zero-shot-classification",
    model="neuralmind/bert-base-portuguese-cased",
    device=0
)

print(f"🤖 Classificando {len(docs_para_classificar)} versículos para a Execução #{exec_id}...")

# 5. Processamento Dinâmico de Métricas XAI, Regras de Negócio e Montagem das Cargas
dados_verso_topico = []             # Tabela principal (Sumarizada)
dados_verso_topico_prob = []        # Tabela detalhada (N:N de probabilidades)

print("⚖️ Calculando métricas XAI dinâmicas e aplicando regras de decisão por gênero...")

for i, row in tqdm(df_input.iterrows(), total=len(df_input), desc="Processando Versículos", unit="v"):
    verso_id = int(row['verso_id'])
    texto_orig = docs_para_classificar[i]
    texto_limpo = row['texto_limpo']
    gen_id = row['genero_id']
    t_len = len(row['texto'])

    # Extração das probabilidades do modelo Zero-Shot
    if texto_orig == "vazio" or texto_limpo.startswith("RUIDO_"):
        # Se for ruído estrutural da Célula 4, atribui distribuição uniforme
        probs_dict = {desc: 1.0 / num_topicos for desc in descricoes_eixos}
    else:
        # Avalia o texto original contra todos os rótulos conceituais
        res = classifier(texto_orig, candidate_labels=descricoes_eixos, multi_label=False)
        probs_dict = dict(zip(res['labels'], res['scores']))

    # Monta o vetor N-dimensional ordenado conforme topicos_desc_ids
    probs_verso = np.array([probs_dict[desc] for desc in descricoes_eixos])

    # GRAVAÇÃO DETALHADA N:N (Popula verso_topico_probabilidade com todos os N tópicos)
    for idx_topico, prob in enumerate(probs_verso):
        dados_verso_topico_prob.append((
            verso_id,
            int(exec_id),
            int(topicos_desc_ids[idx_topico]),
            float(prob)
        ))

    # Métricas XAI dinâmicas
    probs_ordenadas = np.sort(probs_verso)[::-1]
    gap = float(probs_ordenadas[0] - probs_ordenadas[1])
    entropia = float(-np.sum(probs_verso * np.log(probs_verso + 1e-9)))

    # Separa os eixos existenciais do eixo de descarte
    indices_existenciais = [idx for idx in range(num_topicos) if idx != idx_descarte_matriz]
    probs_existenciais = probs_verso[indices_existenciais]

    # Melhor eixo existencial
    best_sub_idx = np.argmax(probs_existenciais)
    best_idx = indices_existenciais[best_sub_idx]
    best_score = float(probs_verso[best_idx])

    p_descarte = float(probs_verso[idx_descarte_matriz])
    margem = float(best_score - p_descarte)

    # --- Lógica Dinâmica de Decisão ---

    # Regra 1: Curto-circuito de ruído estrutural da Célula 4
    if texto_limpo.startswith("RUIDO_"):
        decisao_idx = idx_descarte_matriz
        status = f"Filtro {texto_limpo}"

    # Regra 2: Filtro de brevidade
    elif t_len < 35 and margem < 0.25:
        decisao_idx = idx_descarte_matriz
        status = "Filtro Brevidade"

    # Regra 3: Thresholds Dinâmicos por Gênero Literário
    else:
        if gen_id in [1, 2]: # Pentateuco/Histórico
            if best_score > 0.88 and margem > 0.15:
                decisao_idx, status = best_idx, "Rigor Máximo"
            else:
                decisao_idx, status = idx_descarte_matriz, "Descarte Histórico"

        elif gen_id in [3, 4]: # Poético/Profético
            if best_score > 0.50:
                decisao_idx, status = best_idx, "Sensibilidade Poética"
            else:
                decisao_idx, status = idx_descarte_matriz, "Descarte Poético"

        elif gen_id in [5, 6]: # Evangelhos/Epístolas
            if best_score > 0.60 or (best_score > 0.45 and margem > 0.10):
                decisao_idx, status = best_idx, "Resgate/Consolo"
            else:
                decisao_idx, status = idx_descarte_matriz, "Descarte Didático"

        else: # Geral
            if best_score > 0.75:
                decisao_idx, status = best_idx, "Padrão Geral"
            else:
                decisao_idx, status = idx_descarte_matriz, "Descarte Geral"

    # Captura o topico_descricao_id vitorioso
    topico_desc_id_vitorioso = topicos_desc_ids[decisao_idx]
    score_vitorioso = best_score if decisao_idx != idx_descarte_matriz else p_descarte

    # Tupla para a tabela sumarizada verso_topico
    dados_verso_topico.append((
        verso_id,
        int(exec_id),
        int(topico_desc_id_vitorioso),
        float(score_vitorioso),
        float(margem),
        str(status),
        float(entropia),
        float(gap)
    ))

# 6. Persistência Dupla via UPSERT SQL
cursor = conn.cursor()

print(f"💾 Persistindo {len(dados_verso_topico)} decisões em 'verso_topico'...")
query_verso_topico = """
    INSERT INTO verso_topico (
        verso_id, execucao_pipeline_id, topico_descricao_id,
        similaridade_final, margem_dominancia, status_decisao,
        entropia, gap_confianca
    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    ON CONFLICT(verso_id, execucao_pipeline_id) DO UPDATE SET
        topico_descricao_id = excluded.topico_descricao_id,
        similaridade_final = excluded.similaridade_final,
        margem_dominancia = excluded.margem_dominancia,
        status_decisao = excluded.status_decisao,
        entropia = excluded.entropia,
        gap_confianca = excluded.gap_confianca;
"""
cursor.executemany(query_verso_topico, dados_verso_topico)

print(f"💾 Persistindo {len(dados_verso_topico_prob)} probabilidades individuais em 'verso_topico_probabilidade'...")
query_verso_topico_prob = """
    INSERT INTO verso_topico_probabilidade (
        verso_id, execucao_pipeline_id, topico_descricao_id, probabilidade
    ) VALUES (?, ?, ?, ?)
    ON CONFLICT(verso_id, execucao_pipeline_id, topico_descricao_id) DO UPDATE SET
        probabilidade = excluded.probabilidade;
"""
cursor.executemany(query_verso_topico_prob, dados_verso_topico_prob)

conn.commit()
conn.close()

print(f"✨ Célula 5 concluída! Decisões e detalhamento de probabilidades salvos com sucesso para a execução #{exec_id}.")

⚠️ A Célula 6, que processa a análise de sentimento, é a mais demorada do pipeline. Pode levar mais de 1h até o término.

In [ ]:
# Célula 6: Análise de Sentimento Contextual e Cruzamento Existencial
from pysentimiento import create_analyzer
import pandas as pd
import sqlite3
from tqdm.auto import tqdm

# 1. Validação do Contexto da Execução Atual
try:
    exec_id = EXECUCAO_PIPELINE_ID
except NameError:
    raise RuntimeError("❌ EXECUCAO_PIPELINE_ID não encontrada. Execute a Célula 3B antes de rodar a Célula 6.")

# 2. Inicializar o Analisador de Sentimento
print("🚀 Carregando modelo Transformer para Sentimento (PT-BR)...")
analyzer = create_analyzer(task="sentiment", lang="pt")

# 3. Busca do Texto Original Filtrado pela Execução Corrente
conn = sqlite3.connect(DB_PATH)
query_input = """
    SELECT v.id AS verso_id, v.texto
    FROM verso v
    JOIN verso_topico vt ON v.id = vt.verso_id
    WHERE vt.execucao_pipeline_id = ?;
"""
df_input = pd.read_sql_query(query_input, conn, params=(exec_id,))

textos = df_input['texto'].tolist()
verso_ids = df_input['verso_id'].tolist()

# 4. Execução da Análise em Lotes
print(f"📊 Analisando carga emocional de {len(textos)} versículos para a Execução #{exec_id}...")
dados_insercao = []
batch_size = 64
mapa_num = {'POS': 1, 'NEU': 0, 'NEG': -1}

for i in tqdm(range(0, len(textos), batch_size), desc="Processando Sentimentos"):
    lote = textos[i:i + batch_size]
    ids_lote = verso_ids[i:i + batch_size]
    preds_lote = analyzer.predict(lote)

    for idx, p in enumerate(preds_lote):
        label_str = str(p.output)
        dados_insercao.append((
            int(ids_lote[idx]),
            int(exec_id),
            label_str,
            int(mapa_num.get(label_str, 0)),
            float(p.probas.get('POS', 0)),
            float(p.probas.get('NEG', 0)),
            float(p.probas.get('NEU', 0))
        ))

# 5. Persistência via UPSERT SQL
try:
    cursor = conn.cursor()

    query_upsert = """
        INSERT INTO verso_sentimento (
            verso_id, execucao_pipeline_id, label, sentimento_num,
            score_pos, score_neg, score_neu
        ) VALUES (?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(verso_id, execucao_pipeline_id) DO UPDATE SET
            label = excluded.label,
            sentimento_num = excluded.sentimento_num,
            score_pos = excluded.score_pos,
            score_neg = excluded.score_neg,
            score_neu = excluded.score_neu;
    """

    cursor.executemany(query_upsert, dados_insercao)
    conn.commit()
    print(f"\n✅ Célula 6 concluída! {len(dados_insercao)} sentimentos salvos em 'verso_sentimento' para a execução #{exec_id}.")

    # 6. RESULTADO FINAL: DIAGNÓSTICO VS. ANTÍDOTO (Filtrado por Execução)
    print("\n📈 RESUMO EXECUTIVO: PROBLEMÁTICA (CRISE) VS. ANTÍDOTO (CURA)")

    query_resumo = """
        SELECT
            t.nome AS Eixo_Filosofico,
            COUNT(*) AS Total_Versos,
            SUM(CASE WHEN vs.sentimento_num = 1 THEN 1 ELSE 0 END) AS Antidotos_Cura,
            SUM(CASE WHEN vs.sentimento_num = -1 THEN 1 ELSE 0 END) AS Problematica_Crise,
            ROUND(AVG(vs.sentimento_num), 3) AS Polaridade_Media
        FROM verso_topico vt
        JOIN topico_descricao td ON vt.topico_descricao_id = td.id
        JOIN topico t ON td.topico_id = t.id
        JOIN verso_sentimento vs ON vt.verso_id = vs.verso_id AND vt.execucao_pipeline_id = vs.execucao_pipeline_id
        WHERE vt.execucao_pipeline_id = ? AND t.id != 4
        GROUP BY t.id, t.nome
        ORDER BY Polaridade_Media DESC;
    """

    res_final = pd.read_sql_query(query_resumo, conn, params=(exec_id,))

    # Exibe a tabela no Jupyter/Colab
    display(res_final)

except Exception as e:
    print(f"❌ Erro na persistência de sentimentos: {e}")
finally:
    conn.close()

⚠️ Ao final do processamento, uma das formas para consultar os dados persistidos no banco SQLite, no arquivo **base-dados.db**, é baixar o arquivo do Google Drive, armazenad na pasta **\data** para a máquina local e utilizar uma ferramenta/console que possibilite a execução de consultas SQL SELECT aos dados nas tabelas que armazenam o processamento. Ou, construir scripts Phyton que acessem o banco de dados e gerem tabelas ou gráficos para a análise dos dados.